First we sanity check the corpus...
Let's look at the stories generated and visually ensure they make sense.

In [2]:
import random
import textwrap
import sys

from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

CORPUS_DIR = PROJECT_ROOT / "datasets" / "qwen-emotion-stories" / "corpus"

from core.shards import read_shards
from core.utils import emotion_words_named

In [3]:
from collections import Counter

rows = read_shards(CORPUS_DIR, sample=False)
print(f"{len(rows)} stories")
print(Counter(r["emotion"] for r in rows))

807 stories
Counter({'neutral': 98, 'surprised': 91, 'ashamed': 68, 'disgusted': 65, 'desperate': 64, 'afraid': 59, 'sad': 59, 'calm': 58, 'angry': 54, 'proud': 52, 'excited': 50, 'joyful': 48, 'lonely': 41})


In [4]:
def show_samples(n: int = 2, seed: int | None = None, emotion: str | None = None, sample: bool = True, prompts: bool = True) -> None:
    rows = read_shards(CORPUS_DIR, sample=sample)
    if emotion:
        rows = [r for r in rows if r["emotion"] == emotion]

    for row in random.Random(seed).sample(rows, min(n, len(rows))):
        named = emotion_words_named(row["text"])
        print("=" * 100)
        print(f"{row['emotion']} | {row['topic']} | #{row['index']} | {len(row['text'].split())} words | ends with: {row['text'].rstrip()[-1]!r} | names: {named or 'none'}")
        if prompts:
            print("-" * 100)
            print(textwrap.indent(row["prompt"], "  "))
        print("-" * 100)
        print(textwrap.fill(row["text"], width=88))
        print()

show_samples(2, sample=False)

lonely | Someone's childhood imaginary friend appears in their niece's drawings | #1 | 88 words | ends with: '.' | names: none
----------------------------------------------------------------------------------------------------
  Write a short story (roughly one paragraph) based on the following premise.

  Topic: Someone's childhood imaginary friend appears in their niece's drawings

  The story should follow a character who is feeling lonely.

  Write the story in English. Use either third-person or first-person narration.

  Write between 90 and 130 words. Finish the final sentence. Do not write a title.

  The character is ALREADY feeling lonely in the very first sentence. Open inside the scene, at the moment the feeling is strongest. Do not begin with backstory, scene-setting, or a build-up towards the feeling, and do not begin with "Once upon a time".

  ONE STATE ONLY: the character feels lonely and nothing else, from the first word to the last. Nothing in the story relieves, re

Next up we need to extract activations from all layers to do our difference of means probe

In [5]:
## get activations

from core.models import Model

m = Model()
m.load_weights()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
BATCH_SIZE = 64
DEVICE = m.device

In [7]:
import torch

from collections import defaultdict
from tqdm.auto import tqdm

# Extract activations
# Create a function for extracting activations
# The function should take in a model and a list of texts, and it should return a list of tensors of activations
# For each prompt, there should be an activation with the shape of the residual stream
# Split the texts into train and test before passing into the function - only pass train

SKIP = 18

def extract_activations(texts: list[str], batch_size: int = BATCH_SIZE, layers: list[int] = None, skip: int = SKIP):
    # iterate over texts (they will be specific to an emotion)
    # for each text, teacher force into model and get hidden states
    # get hidden states in each layer and mean-pool them
    # return activations
    m.tok.padding_side = "right"
    final_states = defaultdict(list)

    if layers is None:
        layers = list(range(m.model.config.num_hidden_layers + 1))

    for b in tqdm(range(0, len(texts), batch_size), desc="extract activations", leave=True):
        inputs = m.tok(texts[b : b + batch_size], padding=True, return_tensors='pt', truncation=False).to(DEVICE)

        with torch.no_grad():
            outputs = m.model.model(**inputs, output_hidden_states=True)
        
        mask = inputs.attention_mask.unsqueeze(-1)
        mask[:, :SKIP, :] = 0 # skip first few tokens
        # shape: b, seq, 1

        for l in layers:
            hidden_states = (outputs.hidden_states[l] * mask).sum(1) / mask.sum(1)
            # shape: b, d_model
            
            final_states[l].append(hidden_states)

    return {l: torch.cat(v, dim=0) for l, v in final_states.items()}

activations = extract_activations([r["text"] for r in rows[:BATCH_SIZE]])
activations[0].shape


extract activations:   0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([64, 896])

In [8]:
# Emotion vectors
# Create a function for creating emotion vectors
# It should take an emotion and all the texts, and it should spit out the emotion vector
# It should take the mean of all the train activations for each emotion - e_mean
# It should take the mean of all activations completely - mean
# It should subtract e_mean - mean to find the emotion vector

LAYERS = [24] # arbitrary start

def emotion_vectors(texts: list[str], batch_size: int = BATCH_SIZE, layers: list[int] = LAYERS):
    # for each emotion (apart from neutral)
        # get all the train texts for the specific emotion
        # get activations for those texts
        # store mean activations for emotion
    # subtract out mean of all emotions
    # return list of emotion vectors
    counts = Counter(r["emotion"] for r in rows)
    emotions = sorted(e for e in counts if e != "neutral")
    vectors = {}

    by_emotion = {e: [i for i, t in enumerate(texts) if t["emotion"] == e] for e in emotions}
    acts = extract_activations([t["text"] for t in texts], batch_size=batch_size, layers=layers)
    global_means = {}

    for l in layers:
        class_means = torch.stack([acts[l][by_emotion[e]].mean(0) for e in emotions])
        global_means[l] = class_means.mean(0)
        for i, e in enumerate(emotions):
            # shape: d_model
            vectors[(e, l)] = class_means[i] - global_means[l]

    return vectors, global_means

train = [r for r in rows if r["split"] == "train" and r["emotion"] != "neutral"]
emovecs, global_means = emotion_vectors(train, batch_size=BATCH_SIZE)

extract activations:   0%|          | 0/9 [00:00<?, ?it/s]

In [9]:
emovecs[('afraid', LAYERS[0])].shape

torch.Size([896])

In [10]:
# Denoising
# Create a function to denoise the emotion vectors in line with the paper
# To denoise do PCA on the set of the neutral activations
# Then after that, take the top components explaining 50% of variance and project them out from the emotion vectors
# Fit with PCA once per layer
# How do I confirm they have been denoised?? - 
    # Test with pairwise cosine similarity before and after. Also test with nearest centroid accuracy on held-out topics before vs after

def denoise(vecs: torch.Tensor, neutrals: list[str], batch_size: int = BATCH_SIZE, frac = 0.5, layers: list[int] = None):
    # teacher force the neutrals to get their activations
    # do PCA on the neutral activations 
    # measure variance using eigenvalues as weights
    # take the first n vectors until 0.5 of eigenvalue density
    # project the neutral activations onto those n vectors (to get the right magnitude of noise)
    # do the above with a matrix multiplication
    # subtract out the projected neutral activations from the emotion vectors
    # return the emotion vectors
    emotions = sorted(list(set([e for (e, l) in vecs.keys()])))
    acts = extract_activations(neutrals, batch_size)
    denoised = {}

    if layers is None:
        layers = range(len(acts))

    for l in layers:
        X = acts[l].cpu().float()
        # shape: n, d_model

        Xc = X - X.mean(0)
        _, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
        ratio = (S ** 2) / (S ** 2).sum()
        # shape: n

        k = int((ratio.cumsum(dim=0) < frac).sum().item()) + 1
        pcs = Vh[:k].to(DEVICE)

        for e in emotions:
            v = vecs[(e, l)]
            denoised[(e, l)] = v - ((v @ pcs.T) @ pcs)
        
    return denoised

neutrals = [r["text"] for r in rows if r["emotion"] == "neutral"]
denoised_emovecs = denoise(emovecs, neutrals, layers = LAYERS)

extract activations:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
emotions = sorted(list(set([e for (e, l) in denoised_emovecs.keys()])))
emotions

['afraid',
 'angry',
 'ashamed',
 'calm',
 'desperate',
 'disgusted',
 'excited',
 'joyful',
 'lonely',
 'proud',
 'sad',
 'surprised']

In [12]:
# sanity check denoising
# check that cosine similarity of noisy and noiseless vectors are not close to 0 or 1.0
# if cosine similarity is close to 1.0, then the denoising did little
# if cosine similarity is close to 0.0, then there's a problem with the denoising, or the original vectors were completely noise

cosines = []
for key in denoised_emovecs.keys():
    cosines.append(torch.nn.functional.cosine_similarity(emovecs[key], denoised_emovecs[key], dim=-1))

sum(cosines) / len(cosines)

tensor(0.8630, device='mps:0')

In [ ]:
# logit lens
# the basic idea of the logit lens is to apply an unembedding matrix to the output
# the unembedding matrix should output high probabilities on tokens that match the emotion concept
# this is a crude verification technique but should work

START_LAYER = 10

def logit_lens(vecs: torch.Tensor, k: int = 10) -> torch.Tensor:
    # first thing is to get the unembedding matrix... how? - can use the models one but double-check that's correct
    # HOLD ON - this should take the vectors, not the activations!!!
    # activations: n, seq, d_model, vecs: d_model
    # apply unembedding matrix to each vector for each emotion, layer combination
    # interesting... how to figure out which layer is the right layer??? - analysis to be done here
    # right now we just un-embed all layers after middle
    # decode tokens from logits{}
    # print out top_n most probable tokens per layer

    W_U = m.model.get_output_embeddings().weight
    # shape: vocab, d_model
    # norm = m.model.model.norm

    top_k = defaultdict(list)
    for (e, l), v in vecs.items():
        if l < START_LAYER: # start from the mid-layers
            continue

        h = v.to(W_U.device, W_U.dtype)
        logits = h @ W_U.T
        # shape: vocab
        top_k_ids = logits.topk(k).indices
        top_k_tokens = [m.tok.decode(i) for i in top_k_ids]
        top_k[e].insert(0, ",".join(top_k_tokens))
    
    return top_k
        
lens = logit_lens(denoised_emovecs)

In [54]:
import pandas as pd

pp_lens = {}
for e, tokens in lens.items():
    pp_lens[e] = tokens

df = pd.DataFrame(pp_lens)
print("Layer 25")
df.head(10)

Layer 25


,afraid,angry,ashamed,calm,desperate,disgusted,excited,joyful,lonely,proud,sad,surprised
0,"拼命,剧烈, panic,救命, desperately,Input, panicked,紧...","报复, dumps,睚,头痛,杀人, crap,报仇,暴力,rage,骂","worst, surgeries, painful, uncomfortable, wor...","peaceful, gentle, tranqu,宁静, serene, soothing...","报废,死刑,绝望, Worst,狠,硬, DEAD, Worse,不能再, worse","秽,臭,恶心, vom, disgusting,呕吐, gag,恶,腥,厉","兴奋,激动, excitement,怦,惊喜, thrilling,勇, thrilled,...","happily, sunshine,美好生活, enchant,浪漫,bows, deli...","寂寞, loneliness, silence,寂, lonely, empt,寂静, em...","自信,自豪, proudly, celebrated, proud, empowered, ...","寂寞, melanch,ickness, mourn, mourning,悲伤,侵蚀,悲哀,...","”,,”\n,”:,!”,.”\n,”,[…,问我,可疑, ”"


In [15]:
import pandas as pd

df = pd.DataFrame(rows)[["topic", "emotion", "split"]]
display(pd.crosstab(df.emotion, df.split, margins=True))
display(pd.crosstab(df.topic, df.split, margins=True))

# we can see that the dataset is already split across test and train
# emotions are roughly split accoring to the test train split - 25/75
# topics are split so that there are specific topics for testing and topics for training

split,test,train,All
emotion,,,
afraid,17,42,59
angry,12,42,54
ashamed,19,49,68
calm,17,41,58
desperate,20,44,64
disgusted,16,49,65
excited,12,38,50
joyful,11,37,48
lonely,10,31,41


split,test,train,All
topic,,,
A chef receives a harsh review from a food critic,0,9,9
A coach has to cut a player from the team,8,0,8
A college student discovers their roommate has been reading their journal,8,0,8
A family member announces they're converting to a different religion,0,7,7
A family member wants to sell a cherished heirloom,0,8,8
...,...,...,...
Two siblings discover different versions of their inheritance,0,9,9
Two siblings inherit their grandmother's house,0,9,9
Two strangers discover they share the same rare medical condition,0,5,5


In [16]:
# six-way accuracy - 1

# Here we make a probe predict the emotion concept of an activation
# Then we create a dataset of train and test activations (split by topics)
# Then we check the accuracy of the probe on a held out test set of topics for an arbitrary layer

import torch
import torch.nn.functional as F

emotions = sorted(list(set([e for (e, l) in denoised_emovecs.keys()])))
layers = sorted(list(set([l for (e, l) in denoised_emovecs.keys()])))

layer_vecs = defaultdict(list)
for l in layers:
    for e in emotions:
        layer_vecs[l].append(denoised_emovecs[(e, l)])

# sanity check random vectors have the right shape
CHECK_LAYER = 24
print(layer_vecs[CHECK_LAYER][0].shape)

class EmoProbe(torch.nn.Module):
    def __init__(self, directions: torch.Tensor, global_means: dict):
        # here I need to store the directions as parameters
        # n is the number of emotion directions (vectors), d_model is the size of the vector
        super().__init__()
        self.directions = torch.nn.Buffer(F.normalize(directions, dim=-1))
        # n, d_model
        self.global_means = global_means

    def forward(self, X: torch.Tensor):
        # matmul X with the directions
        # X shape: n, d_model
        # directions shape: n_emotions, d_model
        # out: n, n_emotions
        Xc = X - self.global_means
        return F.normalize(Xc, dim=-1) @ self.directions.T
    
    def pred(self, X: torch.Tensor):
        out = self(X)
        # shape: n, n_emotions
        return out.argmax(-1)

test_rows = [r for r in rows if r["split"] == "test" and r["emotion"] != "neutral"]
test_texts = [r["text"] for r in test_rows]
X_test = extract_activations(test_texts, layers=LAYERS, batch_size=BATCH_SIZE)
y_test = torch.tensor([emotions.index(r["emotion"]) for r in test_rows], dtype=torch.int64).to(DEVICE)

torch.Size([896])


extract activations:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
CHECK_LAYER = 24

directions = torch.stack(layer_vecs[CHECK_LAYER]) # first layer
# n_emotions, d_model

probe = EmoProbe(directions, global_means[CHECK_LAYER]).to(DEVICE)
predictions = probe.pred(X_test[CHECK_LAYER])
accuracy = (predictions == y_test).sum() / y_test.shape[0]

display(pd.DataFrame({
    "predicted": [emotions[i] for i in predictions.tolist()],
    "actual": [emotions[i] for i in y_test.tolist()]
}))

print("accuracy:", accuracy)
print("aligned:", X_test[CHECK_LAYER].shape[0] == len(test_texts) == len(y_test))
print("NaN rows:", torch.isnan(X_test[CHECK_LAYER]).any(-1).sum().item())   # stories shorter than SKIP=18
print("pred distribution:", torch.bincount(predictions, minlength=12).tolist())
print(emotions)

,predicted,actual
0,afraid,afraid
1,lonely,afraid
2,desperate,afraid
3,lonely,afraid
4,surprised,afraid
...,...,...
183,angry,surprised
184,surprised,surprised
185,afraid,surprised
186,sad,surprised


accuracy: tensor(0.4468, device='mps:0')
aligned: True
NaN rows: 0
pred distribution: [20, 21, 7, 17, 23, 10, 6, 19, 21, 9, 16, 19]
['afraid', 'angry', 'ashamed', 'calm', 'desperate', 'disgusted', 'excited', 'joyful', 'lonely', 'proud', 'sad', 'surprised']


In [ ]:
# six-way accuracy - 2

# scored 45% on accuracy - it's way more than random but less than I expected
# Confusion matix to contextualise the error
# Finally do a layer sweep and visualise to find out which layer is most accurate in predicting the emotion for the held out test set of topics

In [46]:
import pandas as pd

# Label rows with predicted emotions
# Cross-tab rows with predicted and actual emotions

df = pd.DataFrame({
    "predicted": [emotions[i] for i in predictions.tolist()],
    "actual": [emotions[i] for i in y_test.tolist()]
})
cm = pd.crosstab(df.predicted, df.actual, margins=False)
display(cm)

actual,afraid,angry,ashamed,calm,desperate,disgusted,excited,joyful,lonely,proud,sad,surprised
predicted,,,,,,,,,,,,
afraid,7,0,5,0,3,3,0,0,0,0,0,2
angry,0,7,0,0,3,5,1,0,0,0,3,2
ashamed,0,0,3,0,0,1,0,0,0,0,2,1
calm,0,0,0,15,0,0,1,0,1,0,0,0
desperate,3,3,3,0,8,0,0,0,1,0,2,3
disgusted,0,0,1,0,1,6,0,0,0,1,1,0
excited,1,0,0,0,0,0,4,0,0,1,0,0
joyful,0,0,2,0,0,0,3,10,0,3,0,1
lonely,2,0,2,1,3,0,0,0,4,2,5,2


In [ ]:
pct = cm.div(cm.sum(axis=1), axis=0)          # row-normalise -> per-class recall

(pct.style
    .background_gradient(cmap="Greens", axis=None, vmin=0, vmax=1)
    .format("{:.0%}")
    .set_caption(f"row = predicted, col = true - accuracy {cm.values.diagonal().sum() / cm.values.sum():.1%}"))

actual,afraid,angry,ashamed,calm,desperate,disgusted,excited,joyful,lonely,proud,sad,surprised
predicted,,,,,,,,,,,,
afraid,35%,0%,25%,0%,15%,15%,0%,0%,0%,0%,0%,10%
angry,0%,33%,0%,0%,14%,24%,5%,0%,0%,0%,14%,10%
ashamed,0%,0%,43%,0%,0%,14%,0%,0%,0%,0%,29%,14%
calm,0%,0%,0%,88%,0%,0%,6%,0%,6%,0%,0%,0%
desperate,13%,13%,13%,0%,35%,0%,0%,0%,4%,0%,9%,13%
disgusted,0%,0%,10%,0%,10%,60%,0%,0%,0%,10%,10%,0%
excited,17%,0%,0%,0%,0%,0%,67%,0%,0%,17%,0%,0%
joyful,0%,0%,11%,0%,0%,0%,16%,53%,0%,16%,0%,5%
lonely,10%,0%,10%,5%,14%,0%,0%,0%,19%,10%,24%,10%
